In [2]:
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import json


In [3]:
MODEL = 'llama3.2:1b'
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [4]:
#call method from the scaper.py
links = fetch_website_links("https://edwarddonner.com")
#print(links)

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

#print(get_links_user_prompt("https://edwarddonner.com"))

In [7]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

#print(select_relevant_links("https://edwarddonner.com"))
#print(select_relevant_links("https://huggingface.co"))

In [8]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    #print(contents)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


#print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [9]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [10]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    # user_prompt = user_prompt[:1_000] # Truncate if more than 5,000 characters
    return user_prompt

#print(get_brochure_user_prompt("HuggingFace", "https://huggingface.co"))

In [11]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    # print(result)
    display(Markdown(result))


In [12]:
# create_brochure("HuggingFace", "https://huggingface.co")
#create_brochure("ED DONNER", "https://edwarddonner.com")

In [13]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [14]:
stream_brochure("ED DONNER", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:1b
Found 2 relevant links


# ED DONNER: A COMPANY REVOLUTIONIZING AI
==========================================


## About Us

ED Donner is a fast-paced company that combines its love for artificial intelligence (AI) with innovation and creativity. Our mission is to make AI accessible and impactful, enabling people to discover their potential and pursue their reason for being.


### Our CEO & CTO: THE LION AT THE CENTER

[**Ed Donner**, co-founder and CTO of Nebula.io]


Our head honcho is the king (or at least, the most passionate about AI). With a background in technology startups and an impressive collection of successful courses on writing code and experimentations with large language models (LLMs), he's pushing the boundaries of what AI can achieve.


### Our Core Values

Humanity | Creativity | Innovation | Collaboration


We strive to create an environment where our team members feel inspired, empowered, and excited about their work.


## What We Do

### AI Curriculum: Join Us in Our Pursuit of a Better Future

ED Donner offers comprehensive courses on the application of LLMs, covering topics from AI engineering and deployment to agent development using n8n. With over 500,000 enrolled students across 194 countries, our curriculum is designed to help you level up your skills.


| Course Title | Brief Description |
| --- | --- |
| AI Coder: Vibe Coder to Agentic Engineer – RESOURCES | Learn how to craft code and create engaging experiences with LLMs. |
| AI Builder with n8n – Create Agents and Voice Agents | Get hands-on experience building agents, voice agents, and more using our popular framework n8n. |

### Connect & Compete in Our LLM Arena

Join us for a unique gaming-like environment where you'll face off against LLMs in a battle of diplomacy and wit. Discover your hidden talents as we play a series of challenges together.


| Game Title | Brief Description |
| --- | --- |
| Connect Four | Play a game of high-stakes diplomacy with other players, making smart moves to outmaneuver opponents. |
| Outsmart AI | Put your LLM powers to the test against our artificially intelligent opponents in this fast-paced arena. |

### Stay Informed and Engaged


Follow us for updates on new courses, industry insights, and company news:


*Twitter: @edwarddonner
*LinkedIn: linkedin.com/company/EdwardDonner
*Facebook: facebook.com/EDDONNERofficial

### Get in Touch

For all your professional inquiries or feedback:


[ed@edwarddonner.com](mailto:ed@edwarddonner.com)
https://www.edwarddonner.com

By joining ED Donner, you're part of an ecosystem that's pushing the boundaries of what AI can achieve. Join our community and become a part of this exciting journey.


Follow us on social media for more information and behind-the-scenes insights into our company and the world of LLMs.
*Instagram: @edwarddonner
*Twitter: @edwarddonner 
*Facebook: facebook.com/EDDONNERofficial